# 🎬 Smart Video Player UI
Interactive video player with chapter navigation and concept timeline.

**Setup:** Set `VIDEO_PATH` to your video file path or URL, and update the `chapters` list with your JSON data.

In [3]:
import json
from IPython.display import display, HTML

# ── CONFIG ──────────────────────────────────────────────────────────────────
VIDEO_PATH = "Training Data\\Week 01 - Embedded S.mp4"   # ← change to your video file / URL
FILE = open("Output\\chapters.json", "r", encoding = "utf-8")
chapters = json.load(FILE)

# ── Keywords that mark a chapter as LOW-VALUE (greyed on timeline) ───────────
# Chapters whose name OR description contain any of these are flagged as 'skip'
LOW_VALUE_KEYWORDS = [
    "introduction", "intro", "summary", "conclusion",
    "welcome", "housekeeping", "announcements", "admin", "example",
    "recap", "review", "demo", "q&a", "off-topic", "discussion", "Lab Allocation"
]

# ── Helpers ──────────────────────────────────────────────────────────────────
def parse_time(t):
    """Convert HH:MM:SS string to seconds."""
    h, m, s = t.split(":")
    return int(h)*3600 + int(m)*60 + int(s)

def is_low_value(chapter):
    text = (chapter["chapter"] + " " + chapter["description"]).lower()
    return any(kw in text for kw in LOW_VALUE_KEYWORDS)

# ── Enrich chapters ──────────────────────────────────────────────────────────
enriched = []
for ch in chapters:
    s = parse_time(ch["start_time"])
    e = parse_time(ch["end_time"])
    enriched.append({**ch, "start_sec": s, "end_sec": e, "low_value": is_low_value(ch)})

total_duration = max(c["end_sec"] for c in enriched)
chapters_json  = json.dumps(enriched)

# ── Render HTML/CSS/JS ───────────────────────────────────────────────────────
html = f"""
<link href="https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;600&family=IBM+Plex+Sans:wght@300;400;600&display=swap" rel="stylesheet">
<style>
  :root {{
    --bg: #0f0f11;
    --panel: #18181c;
    --border: #2a2a30;
    --accent: #e0291d;
    --text: #e8e8ec;
    --muted: #6b6b78;
    --active-bg: rgba(224,41,29,0.12);
    --hover-bg: rgba(255,255,255,0.04);
    --low-value: #2e2e36;
    --font-mono: 'IBM Plex Mono', monospace;
    --font-sans: 'IBM Plex Sans', sans-serif;
  }}
  #vp-root * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  #vp-root {{
    font-family: var(--font-sans);
    background: var(--bg);
    color: var(--text);
    border-radius: 12px;
    overflow: hidden;
    border: 1px solid var(--border);
    display: flex;
    flex-direction: column;
    max-width: 1100px;
    margin: 0 auto;
    box-shadow: 0 24px 64px rgba(0,0,0,0.6);
  }}
  #vp-label {{
    padding: 10px 18px;
    font-family: var(--font-mono);
    font-size: 11px;
    letter-spacing: 0.15em;
    text-transform: uppercase;
    color: var(--muted);
    border-bottom: 1px solid var(--border);
    display: flex; align-items: center; gap: 8px;
  }}
  #vp-label .dot {{
    width: 7px; height: 7px; border-radius: 50%;
    background: var(--accent);
    animation: pulse 1.8s ease-in-out infinite;
  }}
  @keyframes pulse {{
    0%,100%{{opacity:1;transform:scale(1);}}
    50%{{opacity:0.4;transform:scale(0.7);}}
  }}
  #vp-body {{
    display: grid;
    grid-template-columns: 1fr 320px;
  }}
  #vp-left {{
    display: flex; flex-direction: column;
    border-right: 1px solid var(--border);
  }}
  #vp-video-wrapper {{
    position: relative; background: #000; aspect-ratio: 16/9;
  }}
  #vp-video {{ width:100%; height:100%; display:block; object-fit:contain; }}
  #vp-overlay-chapter {{
    position: absolute; top:14px; left:14px;
    background: rgba(0,0,0,0.72);
    backdrop-filter: blur(6px);
    border: 1px solid rgba(255,255,255,0.08);
    border-radius: 6px;
    padding: 5px 10px;
    font-family: var(--font-mono);
    font-size: 11px; color: var(--text);
    pointer-events: none;
  }}
  /* Timeline */
  #vp-timeline-area {{
    padding: 14px 16px 10px;
    background: var(--panel);
    border-top: 1px solid var(--border);
  }}
  #vp-timeline-track {{
    position: relative; height: 36px;
    border-radius: 6px; background: #1e1e24;
    overflow: visible; cursor: pointer;
    border: 1px solid var(--border);
  }}
  .tl-segment {{
    position: absolute; top:0; bottom:0;
    transition: filter 0.2s;
  }}
  .tl-segment:hover {{ filter: brightness(1.35); }}
  .tl-segment.high-value {{
    background: linear-gradient(180deg,#e0291d 0%,#a01d15 100%);
    opacity: 0.88;
  }}
  .tl-segment.low-value {{
    background: var(--low-value); opacity: 0.55;
  }}
  #vp-playhead {{
    position: absolute; top:-4px; bottom:-4px;
    width: 3px; background: #fff; border-radius: 2px;
    pointer-events: none;
    box-shadow: 0 0 8px rgba(255,255,255,0.6);
    z-index: 10; transition: left 0.1s linear;
  }}
  #vp-tl-labels {{
    display: flex; justify-content: space-between;
    margin-top: 6px;
    font-family: var(--font-mono); font-size: 10px; color: var(--muted);
  }}
  #vp-tl-legend {{
    display: flex; gap: 16px; margin-top: 8px;
    font-size: 10px; font-family: var(--font-mono); color: var(--muted);
  }}
  .legend-dot {{
    width: 10px; height: 10px; border-radius: 2px;
    display: inline-block; margin-right: 5px; vertical-align: middle;
  }}
  /* Controls */
  #vp-controls {{
    display: flex; align-items: center; gap: 10px;
    padding: 10px 16px;
    background: var(--panel); border-top: 1px solid var(--border);
  }}
  .ctrl-btn {{
    background: none; border: 1px solid var(--border);
    color: var(--text); border-radius: 6px;
    width: 34px; height: 34px; cursor: pointer; font-size: 14px;
    display: flex; align-items: center; justify-content: center;
    transition: background 0.15s, border-color 0.15s;
  }}
  .ctrl-btn:hover {{ background: var(--hover-bg); border-color: #444; }}
  .ctrl-btn.primary {{
    background: var(--accent); border-color: var(--accent);
    width: 40px; height: 40px; font-size: 16px;
  }}
  .ctrl-btn.primary:hover {{ background: #c02218; }}
  #vp-time {{
    font-family: var(--font-mono); font-size: 12px;
    color: var(--muted); margin-left: auto;
  }}
  #vp-time span {{ color: var(--text); }}
  #vp-vol {{ width: 70px; accent-color: var(--accent); cursor: pointer; }}
  #vp-speed {{
    background: var(--panel); border: 1px solid var(--border);
    color: var(--text); border-radius: 5px;
    padding: 3px 6px; font-family: var(--font-mono);
    font-size: 11px; cursor: pointer;
  }}
  /* Chapter panel */
  #vp-right {{
    display: flex; flex-direction: column; background: var(--panel);
  }}
  #vp-chapters-header {{
    padding: 14px 16px 10px;
    border-bottom: 1px solid var(--border);
    font-size: 11px; font-family: var(--font-mono);
    letter-spacing: 0.12em; text-transform: uppercase;
    color: var(--muted);
    display: flex; align-items: center; justify-content: space-between;
  }}
  #vp-chapters-header em {{ font-style: normal; color: var(--text); font-weight: 600; }}
  #vp-chapters-list {{
    flex: 1; overflow-y: auto;
    scrollbar-width: thin; scrollbar-color: var(--border) transparent;
  }}
  .ch-item {{
    display: flex; align-items: stretch;
    cursor: pointer; border-bottom: 1px solid var(--border);
    transition: background 0.15s; position: relative;
  }}
  .ch-item:hover {{ background: var(--hover-bg); }}
  .ch-item.active {{ background: var(--active-bg); }}
  .ch-item.active::before {{
    content: ''; position: absolute; left: 0; top: 0; bottom: 0;
    width: 3px; background: var(--accent); border-radius: 0 2px 2px 0;
  }}
  .ch-index {{
    width: 36px; display: flex; align-items: center; justify-content: center;
    font-family: var(--font-mono); font-size: 10px; color: var(--muted);
    border-right: 1px solid var(--border); flex-shrink: 0;
  }}
  .ch-content {{ flex: 1; padding: 10px 12px; min-width: 0; }}
  .ch-name {{
    font-size: 12px; font-weight: 600; color: var(--text);
    white-space: nowrap; overflow: hidden; text-overflow: ellipsis; line-height: 1.4;
  }}
  .ch-item.low-val .ch-name {{ color: var(--muted); }}
  .ch-desc {{
    font-size: 10px; color: var(--muted); margin-top: 3px;
    display: -webkit-box; -webkit-line-clamp: 2;
    -webkit-box-orient: vertical; overflow: hidden; line-height: 1.5;
  }}
  .ch-meta {{ display: flex; align-items: center; gap: 6px; margin-top: 5px; }}
  .ch-time {{ font-family: var(--font-mono); font-size: 10px; color: var(--muted); }}
  .ch-badge {{
    font-family: var(--font-mono); font-size: 9px;
    padding: 1px 5px; border-radius: 3px; border: 1px solid;
    text-transform: uppercase; letter-spacing: 0.05em;
  }}
  .ch-badge.high {{ background: rgba(224,41,29,0.15); color:#e0291d; border-color:rgba(224,41,29,0.3); }}
  .ch-badge.low  {{ background: #23232a; color: var(--muted); border-color: var(--border); }}
</style>

<div id="vp-root">
  <div id="vp-label">
    <span class="dot"></span>
    Smart Video Player &nbsp;&middot;&nbsp; Concept Timeline
  </div>
  <div id="vp-body">
    <div id="vp-left">
      <div id="vp-video-wrapper">
        <video id="vp-video" src="{VIDEO_PATH}" preload="metadata"></video>
        <div id="vp-overlay-chapter">—</div>
      </div>
      <div id="vp-timeline-area">
        <div id="vp-timeline-track">
          <div id="vp-playhead" style="left:0%"></div>
        </div>
        <div id="vp-tl-labels">
          <span>0:00</span><span id="tl-mid"></span><span id="tl-end"></span>
        </div>
        <div id="vp-tl-legend">
          <span><span class="legend-dot" style="background:#e0291d"></span>High-value concept</span>
          <span><span class="legend-dot" style="background:#2e2e36"></span>Low-value / skip</span>
        </div>
      </div>
      <div id="vp-controls">
        <button class="ctrl-btn" id="btn-prev">&#9664;&#9664;</button>
        <button class="ctrl-btn primary" id="btn-play">&#9654;</button>
        <button class="ctrl-btn" id="btn-next">&#9654;&#9654;</button>
        <input id="vp-vol" type="range" min="0" max="1" step="0.05" value="1">
        <select id="vp-speed">
          <option value="0.5">0.5&times;</option>
          <option value="1" selected>1&times;</option>
          <option value="1.25">1.25&times;</option>
          <option value="1.5">1.5&times;</option>
          <option value="2">2&times;</option>
        </select>
        <div id="vp-time"><span id="ct">0:00</span> / <span id="tt">—</span></div>
      </div>
    </div>
    <div id="vp-right">
      <div id="vp-chapters-header">
        Chapter Sections &nbsp;<em id="ch-count"></em>
      </div>
      <div id="vp-chapters-list"></div>
    </div>
  </div>
</div>

<script>
(function(){{
  const CHAPTERS = {chapters_json};
  const TOTAL = {total_duration};
  const video    = document.getElementById('vp-video');
  const playBtn  = document.getElementById('btn-play');
  const prevBtn  = document.getElementById('btn-prev');
  const nextBtn  = document.getElementById('btn-next');
  const volEl    = document.getElementById('vp-vol');
  const speedEl  = document.getElementById('vp-speed');
  const ctEl     = document.getElementById('ct');
  const ttEl     = document.getElementById('tt');
  const playhead = document.getElementById('vp-playhead');
  const track    = document.getElementById('vp-timeline-track');
  const overlay  = document.getElementById('vp-overlay-chapter');
  const chList   = document.getElementById('vp-chapters-list');
  document.getElementById('ch-count').textContent = CHAPTERS.length;

  function fmtTime(s) {{
    s = Math.floor(s);
    const h=Math.floor(s/3600), m=Math.floor((s%3600)/60), ss=s%60;
    if(h>0) return `${{h}}:${{String(m).padStart(2,'0')}}:${{String(ss).padStart(2,'0')}}`;
    return `${{m}}:${{String(ss).padStart(2,'0')}}`;
  }}
  function currentChIdx(t) {{
    for(let i=CHAPTERS.length-1;i>=0;i--) if(t>=CHAPTERS[i].start_sec) return i;
    return 0;
  }}

  const dur = TOTAL||1;
  document.getElementById('tl-end').textContent = fmtTime(dur);
  document.getElementById('tl-mid').textContent = fmtTime(dur/2);
  ttEl.textContent = fmtTime(dur);

  // Build timeline segments
  CHAPTERS.forEach((ch,i) => {{
    const seg = document.createElement('div');
    seg.className = 'tl-segment ' + (ch.low_value?'low-value':'high-value');
    seg.style.left  = (ch.start_sec/dur*100).toFixed(3)+'%';
    seg.style.width = ((ch.end_sec-ch.start_sec)/dur*100).toFixed(3)+'%';
    seg.title = ch.chapter+' – '+fmtTime(ch.start_sec);
    seg.addEventListener('click', e=>{{ e.stopPropagation(); video.currentTime=ch.start_sec; }});
    track.appendChild(seg);
  }});
  track.addEventListener('click', e=>{{
    const r=track.getBoundingClientRect();
    video.currentTime=((e.clientX-r.left)/r.width)*dur;
  }});

  // Build chapter list
  CHAPTERS.forEach((ch,i) => {{
    const item = document.createElement('div');
    item.className = 'ch-item'+(ch.low_value?' low-val':'');
    item.dataset.idx = i;
    item.innerHTML = `
      <div class="ch-index">${{i+1}}</div>
      <div class="ch-content">
        <div class="ch-name">${{ch.chapter}}</div>
        <div class="ch-desc">${{ch.description}}</div>
        <div class="ch-meta">
          <span class="ch-time">${{ch.start_time.slice(3)}}</span>
          <span class="ch-badge ${{ch.low_value?'low':'high'}}">${{ch.low_value?'skip':'key'}}</span>
        </div>
      </div>`;
    item.addEventListener('click',()=>{{ video.currentTime=ch.start_sec; video.play(); }});
    chList.appendChild(item);
  }});

  // Controls
  playBtn.addEventListener('click',()=> video.paused?video.play():video.pause());
  video.addEventListener('play', ()=>{{ playBtn.innerHTML='&#10074;&#10074;'; }});
  video.addEventListener('pause',()=>{{ playBtn.innerHTML='&#9654;'; }});
  prevBtn.addEventListener('click',()=>{{
    const idx=currentChIdx(video.currentTime);
    video.currentTime=CHAPTERS[Math.max(0,idx-1)].start_sec;
  }});
  nextBtn.addEventListener('click',()=>{{
    const idx=currentChIdx(video.currentTime);
    video.currentTime=CHAPTERS[Math.min(CHAPTERS.length-1,idx+1)].start_sec;
  }});
  volEl.addEventListener('input',()=>{{ video.volume=volEl.value; }});
  speedEl.addEventListener('change',()=>{{ video.playbackRate=parseFloat(speedEl.value); }});

  // Time update
  let lastIdx=-1;
  video.addEventListener('timeupdate',()=>{{
    const t=video.currentTime;
    playhead.style.left=(t/dur*100).toFixed(2)+'%';
    ctEl.textContent=fmtTime(t);
    const idx=currentChIdx(t);
    if(idx!==lastIdx){{
      lastIdx=idx;
      overlay.textContent=CHAPTERS[idx].chapter;
      document.querySelectorAll('.ch-item').forEach(el=>el.classList.remove('active'));
      const a=document.querySelector(`.ch-item[data-idx="${{idx}}"]`);
      if(a){{ a.classList.add('active'); a.scrollIntoView({{block:'nearest',behavior:'smooth'}}); }}
    }}
  }});
  video.addEventListener('loadedmetadata',()=>{{ ttEl.textContent=fmtTime(video.duration||dur); }});
}})();
</script>
"""

display(HTML(html))
print("\u2705 Player rendered. Update VIDEO_PATH to point at your video file.")


✅ Player rendered. Update VIDEO_PATH to point at your video file.
